# NumJa GPU POC - TornadoVM 5.2.0-jdk21 on NVIDIA T4

Runs `bench.tornadopoc.GemmBench` from branch `gsd/phase-05-hardware-abstraction-layer-gpu-poc` and cross-verifies numerical accuracy against NumPy on the Colab Python kernel.
Output is labelled key=value so you can paste it into `docs/05-GPU-POC-RESULTS.md`.

**Runtime setup:** Menu > Runtime > Change runtime type > Hardware accelerator = **T4 GPU** > OS = Ubuntu 22.04.
**Run all:** Runtime > Run all (or Ctrl+F9).

## Step 1 - Verify GPU + Java
Confirms the runtime actually has the T4 (catches the 'selected CPU runtime by mistake' failure mode up front).

In [1]:
import subprocess, os
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout, r.stderr, r.returncode

out, err, rc = run("nvidia-smi | head -10")
print("=== nvidia-smi ==="); print(out)
if rc != 0 or "T4" not in out:
    print("WARNING: T4 not detected.")
    if err: print(err)
print("=== Java ==="); print(run("java -version 2>&1")[0])
print("=== javac ==="); print(run("javac -version 2>&1")[0])

=== nvidia-smi ===
Wed Sep  9 11:19:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |

=== Java ===
[0.016s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/

Expected: `nvidia-smi` lists `Tesla T4`; Java 21.x (TornadoVM 5.2.0-jdk21 needs JDK 21).

## Step 2 - Install JDK 21 + Maven 3.9.15 (skip if both already present)

In [2]:
import urllib.request, tarfile, glob, shutil

def have_java21():
    out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    return '"21' in out or "21." in out

if not have_java21():
    print("Installing JDK 21 ...")
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-21-jdk", shell=True, check=True)
    out, _, _ = run("readlink -f $(which javac) | sed 's:/bin/javac::'")
    os.environ["JAVA_HOME"] = out.strip()
    print(f"JAVA_HOME={out.strip()}")
else:
    print("JDK 21 already present.")

MVN_DIR = "/opt/maven-3.9.15"
if not os.path.isdir(MVN_DIR):
    print("Installing Maven 3.9.15 ...")
    urllib.request.urlretrieve("https://archive.apache.org/dist/maven/maven-3/3.9.15/binaries/apache-maven-3.9.15-bin.tar.gz", "/tmp/mvn.tgz")
    with tarfile.open("/tmp/mvn.tgz", "r:gz") as t: t.extractall("/opt/")
    extracted = sorted(glob.glob("/opt/apache-maven-3.9.15"))[0]
    if extracted != MVN_DIR: shutil.move(extracted, MVN_DIR)
os.environ["PATH"] = f"{MVN_DIR}/bin:" + os.environ.get("PATH", "")
print("Maven:", run("mvn --version")[0].splitlines()[0])

JDK 21 already present.
Installing Maven 3.9.15 ...


/tmp/ipykernel_418/3148099619.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open("/tmp/mvn.tgz", "r:gz") as t: t.extractall("/opt/")


Maven: [0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


## Step 3 - Install TornadoVM 5.2.0-jdk21 SDK
Downloads the SDK tarball, extracts to `/opt/tornadovm`, sets `TORNADO_SDK`.

In [3]:
TORNADO_VERSION = "5.2.0-jdk21"
SDK_DIR = "/opt/tornadovm"
if not os.path.isdir(SDK_DIR):
    print(f"Downloading TornadoVM {TORNADO_VERSION} ...")
    url = f"https://github.com/beehive-lab/TornadoVM/releases/download/v{TORNADO_VERSION}/tornadovm-{TORNADO_VERSION}-cuda-linux-amd64.tar.gz"
    urllib.request.urlretrieve(url, "/tmp/tornado.tgz")
    with tarfile.open("/tmp/tornado.tgz", "r:gz") as t: t.extractall("/opt/")
    cands = sorted(glob.glob("/opt/tornadovm*"))
    if cands[0] != SDK_DIR:
        if os.path.isdir(SDK_DIR): shutil.rmtree(SDK_DIR)
        shutil.move(cands[0], SDK_DIR)
    print("Installed.")
else:
    print(f"{SDK_DIR} already exists.")

os.environ["TORNADO_SDK"] = SDK_DIR
setenv = os.path.join(SDK_DIR, "setenv.sh")
if os.path.isfile(setenv):
    proc = subprocess.run(f"bash -c 'source {setenv} && env'", shell=True, capture_output=True, text=True)
    for line in proc.stdout.splitlines():
        if "=" in line:
            k, _, v = line.partition("="); os.environ[k] = v
print("TORNADO_SDK =", os.environ.get("TORNADO_SDK"))

/tmp/ipykernel_418/3738717496.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open("/tmp/tornado.tgz", "r:gz") as t: t.extractall("/opt/")


Installed.
TORNADO_SDK = /opt/tornadovm


In [ ]:
# Colab ships NVIDIA driver 580.x supporting CUDA 13, but TornadoVM 5.2.0-jdk21 ships
# CUDA 12.x binaries (PTX loader uses cuCtxCreate signature that changed in CUDA 13).
# Workaround: install CUDA 12 toolkit via apt and prepend its lib64 to LD_LIBRARY_PATH
# so the SDK's native libtornado driver resolves against libcuda 12 instead of 13.
import os
if "LD_LIBRARY_PATH" not in os.environ or "cuda-12" not in os.environ.get("LD_LIBRARY_PATH", ""):
    print("Installing CUDA 12.6 toolkit ...")
    subprocess.run("apt-get update -qq", shell=True, check=True)
    subprocess.run("apt-get install -y -qq cuda-toolkit-12-6 2>&1 | tail -3", shell=True, check=True)
    cuda12_lib = "/usr/local/cuda-12.6/lib64"
    if os.path.isdir(cuda12_lib):
        os.environ["LD_LIBRARY_PATH"] = cuda12_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        print(f"LD_LIBRARY_PATH now: {os.environ['LD_LIBRARY_PATH']}")
    else:
        print("WARNING: cuda-12.6 not found at", cuda12_lib, "- check apt repo")
else:
    print("CUDA 12 LD_LIBRARY_PATH already set.")
out, _, _ = run("ls /usr/local/cuda-12.6/lib64/libcuda* 2>&1 | head -5")
print("libcuda libs:", out)


In [4]:
import os
# TornadoVM script checks JAVA_HOME early; ensure it is set before invoking.
if "JAVA_HOME" not in os.environ or not os.environ["JAVA_HOME"]:
    # Try to derive from `which java` if not set.
    out, _, _ = run("readlink -f $(which java) | sed 's:/bin/java::'")
    if out.strip():
        os.environ["JAVA_HOME"] = out.strip()
        print(f"JAVA_HOME inferred: {out.strip()}")
tornado_bin = os.path.join(os.environ.get("TORNADO_SDK", "/opt/tornadovm"), "bin", "tornado")
out, err, rc = run(f"export JAVA_HOME={os.environ.get('JAVA_HOME','')}; {tornado_bin} --devices 2>&1 | head -30")
print(out)
if err.strip(): print("STDERR:", err)


JAVA_HOME inferred: /usr/lib/jvm/java-21-openjdk-amd64
[WARNING] TORNADO_SDK is deprecated, please use TORNADOVM_HOME instead
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

Number of Tornado drivers: 1
Driver: CUDADriver
  Total number of CUDADriver devices  : 1
  Tornado device=0:0  (DEFAULT)
	CUDA --  [NVIDIA CUDA] -- Tesla T4
		Global Memory Size: 14.6 GB
		Local Memory Size: 48.0 KB
		Workgroup Dimensions: 3
		Total Number of Block Threads: [1024]
		Max WorkGroup Configuration: [1024, 1024, 64]
		Device OpenCL C version: CUDA C 1.0





## Step 4 - Clone repo + checkout Phase 5 branch

In [5]:
REPO_DIR = "/content/java_ml"
BRANCH = "gsd/phase-05-hardware-abstraction-layer-gpu-poc"
if os.path.isdir(REPO_DIR):
    print(f"Resetting {REPO_DIR} to {BRANCH}")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", "https://github.com/minhhhduc/jml.git", REPO_DIR], check=True)
head = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print("Repo HEAD:", head)

Repo HEAD: 825b401 fix(05-03): CUDA driver dep + FJP threshold + manual jar install


## Step 5 - Configure Maven repo for TornadoVM SDK + build modules
The SDK bundles a local Maven repo. Find it and write `~/.m2/settings.xml`.

In [6]:
# SDK ships flat jars in $SDK_DIR/share/java/tornado/. The CUDA variant doesn't have a
# bundled Maven repo, so install each jar into ~/.m2/repository with the right coords.
import os, glob
jars = {
    "tornado-api": "tornado-api-5.2.0-jdk21.jar",
    "tornado-runtime": "tornado-runtime-5.2.0-jdk21.jar",
    "tornado-drivers-cuda": "tornado-drivers-cuda-5.2.0-jdk21.jar",
    "tornado-drivers-common": "tornado-drivers-common-5.2.0-jdk21.jar",
}
sdk_jars_dir = os.path.join(SDK_DIR, "share", "java", "tornado")
m2 = os.path.expanduser("~/.m2/repository"); os.makedirs(m2, exist_ok=True)
installed = []
for art, jar_name in jars.items():
    jar_path = os.path.join(sdk_jars_dir, jar_name)
    if not os.path.isfile(jar_path):
        print(f"SKIP {art}: {jar_path} not found")
        continue
    cmd = (f"mvn install:install-file -Dfile={jar_path} "
           f"-DgroupId=io.github.beehive-lab -DartifactId={art} "
           f"-Dversion=5.2.0-jdk21 -Dpackaging=jar -q")
    out, err, rc = run(cmd)
    if rc == 0:
        installed.append(art); print(f"OK   {art}")
    else:
        print(f"FAIL {art}: rc={rc}"); print(err[-800:])
# Also write the parent POM (tornado-drivers:pom:5.2.0-jdk21) directly.
# tornado-drivers-cuda inherits from tornado-drivers packaging=pom; the SDK doesn't ship
# the parent jar, so write a minimal stub.
parent_dir = os.path.join(m2, "io", "github", "beehive-lab", "tornado-drivers", "5.2.0-jdk21")
os.makedirs(parent_dir, exist_ok=True)
parent_pom = '''<?xml version="1.0" encoding="UTF-8"?>
<project xmlns="http://maven.apache.org/POM/4.0.0">
    <modelVersion>4.0.0</modelVersion>
    <groupId>io.github.beehive-lab</groupId>
    <artifactId>tornado-drivers</artifactId>
    <version>5.2.0-jdk21</version>
    <packaging>pom</packaging>
</project>
'''
with open(os.path.join(parent_dir, "tornado-drivers-5.2.0-jdk21.pom"), "w") as f:
    f.write(parent_pom)
print("Wrote parent POM: tornado-drivers:pom:5.2.0-jdk21")
msg = "Installed " + str(len(installed)) + "/" + str(len(jars)) + " jars into " + m2
print(msg)


OK   tornado-api
OK   tornado-runtime
OK   tornado-drivers-cuda
OK   tornado-drivers-common
Installed 4/4 jars into /root/.m2/repository


In [7]:
%cd /content/java_ml
print("=== Install numja-core to local Maven repo ===")
out, err, rc = run("mvn -q -pl modules/numja -am install -DskipTests 2>&1 | tail -10")
print(out if out else "OK (no output)")
if rc != 0 and err: print("ERR:", err[-1500:])

/content/java_ml
=== Install numja-core to local Maven repo ===
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate



In [8]:
%cd /content/java_ml
print("=== Full numja test suite (should be 90/90 green) ===")
out, _, _ = run("mvn -pl modules/numja -am test 2>&1 | grep -E 'Tests run:|BUILD'")
print(out)

/content/java_ml
=== Full numja test suite (should be 90/90 green) ===
[INFO] Tests run: 6, Failures: 0, Errors: 0, Skipped: 0, Time elapsed: 7.168 s -- in com.numja.core.CpuThreadBackendGoldenTest
[INFO] Tests run: 3, Failures: 0, Errors: 0, Skipped: 0, Time elapsed: 0.008 s -- in com.numja.core.BackendSelectorTest
[INFO] Tests run: 1, Failures: 0, Errors: 0, Skipped: 0, Time elapsed: 0 s -- in numja.linalg.LinAlgTest
[INFO] Tests run: 1, Failures: 0, Errors: 0, Skipped: 0, Time elapsed: 0.011 s -- in numja.tests.NumJaJUnitTest
[INFO] Tests run: 8, Failures: 0, Errors: 0, Skipped: 0, Time elapsed: 0.262 s -- in numja.core.ParallelCompensationTest
[INFO] Tests run: 6, Failures: 0, Errors: 0, Skipped: 0, Time elapsed: 0.238 s -- in numja.core.ParallelReduceTest
[INFO] Tests run: 1, Failures: 0, Errors: 0, Skipped: 0, Time elapsed: 0.001 s -- in numja.core.NDArrayTest
[ERROR] Tests run: 4, Failures: 1, Errors: 0, Skipped: 0, Time elapsed: 25.34 s <<< FAILURE! -- in numja.core.ParallelReg

In [9]:
%cd /content/java_ml
print("=== Build POC shaded jar ===")
out, err, rc = run("mvn -q -pl bench/tornado-poc -am package -DskipTests 2>&1 | tail -15")
print(out if out else "OK (no output)")
if rc != 0 and err: print("ERR:", err[-1500:])
jar_path = "/content/java_ml/bench/tornado-poc/target/tornado-poc-jar.jar"
print(f"\nJar: exists={os.path.isfile(jar_path)}" + (f", size={os.path.getsize(jar_path)//1024} KB" if os.path.isfile(jar_path) else ""))

/content/java_ml
=== Build POC shaded jar ===
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[ERROR] Failed to execute goal on project tornado-poc: Could not collect dependencies for project com.numja:tornado-poc:jar:0.1.0
[ERROR] Failed to read artifact descriptor for io.github.beehive-lab:tornado-drivers-cuda:jar:5.2.0-jdk21
[ERROR] 	Caused by: The following artifacts could not be resolved: io.github.beehive-lab:tornado-drivers:pom:5.2.0-jdk21 (absent): Could not find artifact io.github.beehive-lab:tornado-drivers:pom:5.2.0-jdk21 in central (https://repo.maven.apache.org/maven2)
[ERROR] 
[ERROR] -> [Help 1]
[ERROR] 
[ERROR] To see the full stack trace of the errors, re-run Maven with the -e switch.
[ERROR] Re-run Maven using the -X switch to enable full debug logging.
[ERROR] 
[ERROR] For more information about the errors and possible solutions, please read the followin

## Step 6 - Set up shared random input (matches GemmBench.java seededRandom seeds)

In [10]:
import numpy as np
N = int(os.environ.get("BENCH_SIZE", "8192"))
SEED_A, SEED_B = 0xC0FFEE, 0xBADF00D
A_np = np.random.default_rng(SEED_A).random((N, N))
B_np = np.random.default_rng(SEED_B).random((N, N))
print(f"Shape: A={A_np.shape}, B={B_np.shape}, dtype={A_np.dtype}")
print(f"A[0,0]={A_np[0,0]:.18e}, B[0,0]={B_np[0,0]:.18e}")
print(f"A.min()={A_np.min():.6f}, A.max()={A_np.max():.6f}")

Shape: A=(8192, 8192), B=(8192, 8192), dtype=float64
A[0,0]=9.402643811168176491e-01, B[0,0]=7.085682965465434080e-01
A.min()=0.000000, A.max()=1.000000


## Step 7 - Run CPU baseline + GPU on T4 (GemmBench writes 4 error metrics)

In [11]:
JAR = "/content/java_ml/bench/tornado-poc/target/tornado-poc-jar.jar"
%cd /content/java_ml
print("=== CPU baseline (no -Dtornado.device) ===")
out, err, rc = run(f"java -cp {JAR} -Dbench.env=colab -Dbench.size={N} bench.tornadopoc.GemmBench")
print(out)
with open("/tmp/poc-cpu.log", "w") as f: f.write(out)

/content/java_ml
=== CPU baseline (no -Dtornado.device) ===
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate



In [12]:
JAR = "/content/java_ml/bench/tornado-poc/target/tornado-poc-jar.jar"
%cd /content/java_ml
print("=== GPU run on T4 (nvidia:0:0) ===")
# CUDA 12 lib path must precede system CUDA 13 (issue #710).
env_prefix = "export LD_LIBRARY_PATH=/usr/local/cuda-12.6/lib64:\; "
out, err, rc = run(env_prefix + f"java -cp {JAR} -Dtornado.device=nvidia:0:0 -Dbench.env=colab -Dbench.size={N} bench.tornadopoc.GemmBench")
print(out)
if err.strip(): print("STDERR:", err)
with open("/tmp/poc-gpu.log", "w") as f: f.write(out)


/content/java_ml
=== GPU run on T4 (nvidia:0:0) ===
[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

STDERR: Error: Could not find or load main class bench.tornadopoc.GemmBench
Caused by: java.lang.ClassNotFoundException: bench.tornadopoc.GemmBench



## Step 8 - Cross-verify numerical accuracy against NumPy on Colab kernel

We expect the Java CPU path (via EJML) and the Java GPU path (via TornadoVM kernel) to produce the same matrix as `numpy.matmul(A, B)` to within float64 round-off. Frobenius relative error should be ~1e-12 to 1e-14.

In [13]:
import re
def parse_kv(log):
    out = {}
    for line in log.splitlines():
        m = re.match(r"^([a-z_]+)=(.+)$", line.strip())
        if m: out[m.group(1)] = m.group(2)
    return out

gpu_metrics = parse_kv(open("/tmp/poc-gpu.log").read())
cpu_metrics = parse_kv(open("/tmp/poc-cpu.log").read())
print("=== GPU run key=value ===")
for k in ["env","device","jdk","hardware","tornado.device","size",
         "cpu_baseline_ms","gpu_ms","transfer_ms",
         "speedup_ratio","transfer_pct",
         "cpu_vs_gpu_max_abs_err","cpu_vs_gpu_max_rel_err",
         "cpu_vs_gpu_frob_rel_err","cpu_vs_gpu_mae",
         "result","verdict"]:
    if k in gpu_metrics: print(f"  {k} = {gpu_metrics[k]}")

=== GPU run key=value ===


In [14]:
print(f"Computing numpy.matmul({A_np.shape}, {B_np.shape}) ...")
%time C_np = A_np @ B_np
print(f"C_np shape={C_np.shape}, dtype={C_np.dtype}")
frob_ref = np.linalg.norm(C_np, 'fro')
print(f"||C_np||_F = {frob_ref:.6e}")
print(f"max |C_np| = {np.abs(C_np).max():.6e}")
print(f"C_np[0, 0] = {C_np[0, 0]:.18e}")
print(f"C_np[N-1, N-1] = {C_np[N-1, N-1]:.18e}")

Computing numpy.matmul((8192, 8192), (8192, 8192)) ...
CPU times: user 37 s, sys: 262 ms, total: 37.3 s
Wall time: 22.4 s
C_np shape=(8192, 8192), dtype=float64
||C_np||_F = 1.677836e+07
max |C_np| = 2.149594e+03
C_np[0, 0] = 2.082603434080436728e+03
C_np[N-1, N-1] = 2.013594158271614333e+03


In [15]:
# Independent cross-check: how does numpy's matmul compare to the Java CPU/GPU outputs?
# For N=8192 in float64, theoretical error is O(N*eps) ~ 1.8e-11.
frob_np = float(np.linalg.norm(C_np, 'fro'))
print(f"||C_np||_F = {frob_np:.6e}")
print()
print("Threshold rule: result=NUMERIC_MISMATCH if cpu_vs_gpu_frob_rel_err > 1e-9")
print("Expected: ~1e-12 (CPU vs GPU rounding difference of one FMA at N=4096)")
frob_metric = gpu_metrics.get("cpu_vs_gpu_frob_rel_err", "?")
if frob_metric not in ("?", "NA"):
    print(f"GemmBench reported cpu_vs_gpu_frob_rel_err = {frob_metric}")
    val = float(frob_metric)
    if val <= 1e-9: print("PASS: below 1e-9 threshold.")
    else: print("FAIL: above threshold; kernel bug.")

||C_np||_F = 1.677836e+07

Threshold rule: result=NUMERIC_MISMATCH if cpu_vs_gpu_frob_rel_err > 1e-9
Expected: ~1e-12 (CPU vs GPU rounding difference of one FMA at N=4096)


## Step 9 - Decision summary

Capture this block + the GPU `key=value` output above and paste into `docs/05-GPU-POC-RESULTS.md` under `## Colab (NVIDIA T4)`.

In [16]:
verdict = gpu_metrics.get("verdict", "?")
frob = gpu_metrics.get("cpu_vs_gpu_frob_rel_err", "?")
speedup = gpu_metrics.get("speedup_ratio", "?")
transfer_pct = gpu_metrics.get("transfer_pct", "?")
print(f"Auto-verdict:           {verdict}")
print(f"CPU vs GPU Frobenius:   {frob}")
print(f"Speedup (CPU/GPU):      {speedup}")
print(f"Transfer overhead:      {transfer_pct}%")
print()
if verdict == "GO":
    print("GO recommendation: pursue GPU backend milestone in v0.4.0+")
elif verdict == "NO-GO":
    print("NO-GO recommendation: defer GPU backend to v0.5.0+; investigate the bottleneck")
else:
    print("INSUFFICIENT_DATA: re-run with smaller size or fix kernel before deciding")

Auto-verdict:           ?
CPU vs GPU Frobenius:   ?
Speedup (CPU/GPU):      ?
Transfer overhead:      ?%

INSUFFICIENT_DATA: re-run with smaller size or fix kernel before deciding


## Decision rubric

**Numerical (must hold):**
- `cpu_vs_gpu_frob_rel_err <= 1e-9` (otherwise `result=NUMERIC_MISMATCH`, kernel has a bug)
- Expected ~1e-12 for double-precision GEMM at N=4096 (O(N*eps))

**Performance (for go/no-go):**
- `speedup_ratio >= 2.0` AND `transfer_pct < 50.0` -> **GO**
- Otherwise -> **NO-GO**

**Capture format for `docs/05-GPU-POC-RESULTS.md`:**
```
## Colab (NVIDIA T4)

```
env=colab
device=<...>
jdk=<...>
hardware=<...>
cpu_baseline_ms=<...>
gpu_ms=<...>
speedup_ratio=<...>
transfer_pct=<...>
cpu_vs_gpu_max_abs_err=<...>
cpu_vs_gpu_max_rel_err=<...>
cpu_vs_gpu_frob_rel_err=<...>
cpu_vs_gpu_mae=<...>
verdict=<GO|NO-GO>
```

Recommendation: <GO or NO-GO one-liner>
```